# 🛡️ GuardShield AI — Lab 20GB VRAM Training Notebook
### Model: DeBERTa-v3-Large (435M Parameters) across 512-Token Context
**Target Hardware**: Lab Server / Container (20 GB VRAM)
**Primary Task**: Fine-tune High-Capacity Ingress Pre-Scan Classifier on 15,304-Sample Benchmark

---
This notebook executes the complete high-capacity training and evaluation pipeline for GuardShield AI on your 20 GB GPU.

## 🖥️ Step 1: Verify Hardware & GPU VRAM Allocation

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:          {torch.cuda.get_device_name(0)}")
    print(f"Dedicated VRAM:       {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Support:         {torch.cuda.is_bf16_supported()}")

## 📦 Step 2: Clone or Pull Latest GuardShield AI Repository

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists("GuardShield-AI"):
    !git clone https://github.com/Shourya3113/GuardShield-AI.git
    %cd GuardShield-AI
else:
    %cd GuardShield-AI
    !git pull origin main

!pip install -q transformers datasets accelerate sentencepiece scikit-learn pandas numpy

## 📊 Step 3: Verify the 15,304-Sample Benchmark Splits

In [ ]:
import json
from pathlib import Path

manifest_path = Path("data/massive_benchmark/dataset_manifest.json")
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        print("Dataset Manifest:")
        print(json.dumps(json.load(f), indent=2))
else:
    print("Generating massive benchmark splits...")
    !python src/prepare_massive_training_data.py --stage1-samples 10000 --stage2-samples 5000

## 🚀 Step 4: Run High-VRAM DeBERTa-v3-Large Fine-Tuning
* **Backbone**: `microsoft/deberta-v3-large` (435M params)
* **Context Length**: 512 Tokens (extendable to 1024)
* **Memory Protection**: Non-Reentrant Gradient Checkpointing + BF16/FP16 mixed precision
* **Effective Batch Size**: 64 (16 per-device x 4 gradient accumulation steps)
* **Automated Post-Training Holdouts**: In-Distribution (986), XSTest FPR (450), Long-Context (500)

In [ ]:
!python src/train_prescan_large.py \
    --model-name "microsoft/deberta-v3-large" \
    --max-length 512 \
    --batch-size 16 \
    --grad-accum 4 \
    --epochs 3 \
    --output-dir "models/deberta_v3_large_prescan_v4"

## 📈 Step 5: Inspect Empirical Benchmark Summary & Telemetry

In [ ]:
import json
from pathlib import Path

summary_file = Path("models/deberta_v3_large_prescan_v4/training_summary.json")
if summary_file.exists():
    with open(summary_file, "r") as f:
        res = json.load(f)
    print("=" * 70)
    print("  EMPIRICAL TRAINING SUMMARY (DEBERTA-V3-LARGE 435M)")
    print("=" * 70)
    print(f"Model:                {res.get('model_architecture')}")
    print(f"Context Window:       {res.get('context_window')} tokens")
    print(f"Training Time:        {res.get('training_time_seconds')} seconds")
    print(f"Peak VRAM:            {res.get('hardware', {}).get('peak_vram_mb')} MB")
    print("-" * 70)
    print(f"In-Distribution F1:   {res.get('results', {}).get('in_distribution', {}).get('f1_score')}")
    print(f"XSTest Benign FPR:    {res.get('results', {}).get('xstest_fpr', {}).get('false_positive_rate_pct')}%")
    print(f"Long-Context F1:      {res.get('results', {}).get('long_context_ablation', {}).get('f1_score')}")
    print("=" * 70)

## 💾 Step 6: Package Model Weights for Deployment

In [ ]:
!tar -czvf deberta_v3_large_prescan_v4.tar.gz models/deberta_v3_large_prescan_v4/
print("Weights packaged into deberta_v3_large_prescan_v4.tar.gz")